# Phase 5 — Legal Statute Identification (multi-label) · Kaggle 2× T4

**Inputs:** `prefixlm-run1` (tokenizer) + `phase1-models`. Settings: GPU T4 ×2, Internet On, secret `HF_TOKEN`.

**Yours (rule 17.5):** the model file (your Phase-4 `cj_model.py` works unchanged: its head outputs one score per statute) and every `TODO`.
**Harness:** data, token cache, chunking, multi-hot targets, BCE / focal loss, pos_weight option, dev macro-F1 per epoch,
thresholds (fixed / global / per-statute, tuned on dev), min labels per doc, KD plumbing, `[{"id", "pred": [ids]}]` + zip.

Phase-5 technique list (doc 11.2.1): Full FT · Partial FT · QLoRA · **LoRMA** · AdaLoRA · Adapters · Prefix tuning · Prompt tuning · **KD**.
(BitFit is NOT on this list.)

Order: cells 1-6 once (tests pass, data check, SMOKE TEST PASSED) → paste your model file → model check → experiments.

In [2]:
# 1 — setup
import os, json, glob, shutil, time, subprocess, sys
os.chdir("/kaggle/working")
import torch
NGPU = torch.cuda.device_count(); print("GPUs:", NGPU)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("HF_TOKEN secret not available:", e)

def find(pattern):
    return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

GPUs: 2


In [3]:
# 2 — Phase-5 data + tokenizer
from huggingface_hub import hf_hub_download
REPO = "Exploration-Lab/CS-781-Capstone"
P5 = {}
for f in ["Train/train.json", "Train/dev.json", "Test/test.json"]:
    P5[f.split("/")[-1].replace(".json", "")] = hf_hub_download(REPO, f"Phase_5_LSI/{f}", repo_type="dataset")
print(json.dumps(P5, indent=1))
os.makedirs("tokenizer", exist_ok=True)
for fn in ["vocab.json", "merges.txt"]:
    shutil.copy(find(fn)[0], f"tokenizer/{fn}")

Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

{
 "train": "/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_5_LSI/Train/train.json",
 "dev": "/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_5_LSI/Train/dev.json",
 "test": "/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_5_LSI/Test/test.json"
}


In [4]:
%%writefile lsi_data.py
"""lsi_data.py — Phase-5 Legal Statute Identification data handling."""
import hashlib, json, os, pickle
from collections import Counter

import numpy as np

ID_KEYS = ("id", "case_id", "uid", "doc_id", "name")
TEXT_KEYS = ("text", "document", "doc", "judgment", "judgement", "case_text", "content", "facts")
LABEL_KEYS = ("labels", "label", "statutes", "statute_ids", "statute", "targets")
DUPS = Counter()


def _get(rec, keys):
    for k in keys:
        if k in rec:
            return rec[k]
    return None


def _norm_labels(v):
    if v is None:
        return None
    if not isinstance(v, (list, tuple)):
        v = [v]
    return sorted({int(float(x)) for x in v})


def load_cases(path):
    obj = json.load(open(path, encoding="utf-8"))
    if isinstance(obj, dict) and not any(k in obj for k in ID_KEYS):
        for k in ("data", "cases", "examples", "instances"):
            if isinstance(obj.get(k), list):
                obj = obj[k]
                break
    recs = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            if isinstance(v, str):
                recs.append(dict(id=k, text=v))
            else:
                r = dict(v); r.setdefault("id", k); recs.append(r)
    else:
        recs = list(obj)
    out = {}
    for i, r in enumerate(recs):
        cid = _get(r, ID_KEYS)
        cid = i if cid is None else cid
        text = _get(r, TEXT_KEYS)
        assert text is not None, f"record {cid}: no text field among {TEXT_KEYS}; keys {list(r)[:8]}"
        if isinstance(text, (list, tuple)):
            text = "\n".join(str(t) for t in text)
        key, text, lab = str(cid), str(text), _norm_labels(_get(r, LABEL_KEYS))
        if key in out:
            if out[key]["text"] == text and out[key]["labels"] == lab:
                DUPS["identical_dropped"] += 1
                continue
            j = 2
            while f"{key}#{j}" in out:
                j += 1
            key = f"{key}#{j}"
            DUPS["different_kept"] += 1
        out[key] = dict(id=cid, text=text, labels=lab)
    return out


def lowercase_share(cases):
    texts = [v["text"] for v in cases.values()]
    return sum(1 for t in texts if not any(c.isupper() for c in t[:5000])) / max(1, len(texts))


def auto_lowercase(train_cases, test_cases):
    return lowercase_share(test_cases) >= 0.9 and lowercase_share(train_cases) < 0.5


def tokenize_cases(cases, encode_batch, cache_dir=None, tag=""):
    h = hashlib.md5((tag + "|".join(f"{k}:{len(v['text'])}" for k, v in cases.items())).encode()).hexdigest()[:16]
    path = os.path.join(cache_dir, f"lsi_tok_{h}.pkl") if cache_dir else None
    if path and os.path.exists(path):
        return pickle.load(open(path, "rb"))
    keys = list(cases)
    out = {}
    for i in range(0, len(keys), 512):
        ks = keys[i:i + 512]
        for k, ids in zip(ks, encode_batch([cases[k]["text"] for k in ks])):
            out[k] = np.asarray(ids, dtype=np.uint16)
    if path:
        os.makedirs(cache_dir, exist_ok=True)
        tmp = path + f".tmp{os.getpid()}"
        pickle.dump(out, open(tmp, "wb"), protocol=4)
        os.replace(tmp, path)
    return out


def chunk_doc(n_tokens, chunk_len, stride=None, max_chunks=None, select="first"):
    stride = int(stride or chunk_len)
    assert 0 < stride <= chunk_len, "need 0 < stride <= chunk_len"
    if n_tokens <= chunk_len:
        spans = [(0, max(1, n_tokens))]
    else:
        starts = list(range(0, n_tokens - chunk_len + 1, stride))
        if starts[-1] + chunk_len < n_tokens:
            starts.append(n_tokens - chunk_len)
        spans = [(s, s + chunk_len) for s in starts]
    k = len(spans) if not max_chunks else min(int(max_chunks), len(spans))
    if select == "all" or k == len(spans):
        return spans
    if select == "first":
        return spans[:k]
    if select == "last":
        return spans[-k:]
    if select == "head_tail":
        h = (k + 1) // 2
        return spans[:h] + spans[len(spans) - (k - h):] if k - h else spans[:h]
    if select == "spread":
        idx = sorted({round(i * (len(spans) - 1) / max(1, k - 1)) for i in range(k)}) if k > 1 else [0]
        return [spans[i] for i in idx]
    raise ValueError(f"select must be first | last | head_tail | spread | all, got {select!r}")


def _prf(tp, fp, fn):
    p = np.divide(tp, tp + fp, out=np.zeros_like(tp, dtype=float), where=(tp + fp) > 0)
    r = np.divide(tp, tp + fn, out=np.zeros_like(tp, dtype=float), where=(tp + fn) > 0)
    f = np.divide(2 * p * r, p + r, out=np.zeros_like(p), where=(p + r) > 0)
    return p, r, f


def scores(Pb, Y, labels=None):
    Pb, Y = np.asarray(Pb, bool), np.asarray(Y, bool)
    tp = (Pb & Y).sum(0).astype(float); fp = (Pb & ~Y).sum(0).astype(float); fn = (~Pb & Y).sum(0).astype(float)
    p, r, f = _prf(tp, fp, fn)
    sup = Y.sum(0)
    present = sup > 0
    mp, mr, mf = _prf(np.array([tp.sum()]), np.array([fp.sum()]), np.array([fn.sum()]))
    out = dict(macro_f1=float(f.mean()), macro_f1_present=float(f[present].mean()) if present.any() else 0.0,
               macro_p_present=float(p[present].mean()) if present.any() else 0.0,
               macro_r_present=float(r[present].mean()) if present.any() else 0.0,
               micro_f1=float(mf[0]), micro_p=float(mp[0]), micro_r=float(mr[0]),
               subset_acc=float((Pb == Y).all(1).mean()), n=int(len(Y)), labels_present=int(present.sum()),
               avg_pred_labels=float(Pb.sum(1).mean()), avg_gold_labels=float(Y.sum(1).mean()),
               empty_pred_docs=int((Pb.sum(1) == 0).sum()))
    if labels is not None:
        out["per_label"] = {int(l): dict(f1=float(f[i]), p=float(p[i]), r=float(r[i]), support=int(sup[i]),
                                         predicted=int(tp[i] + fp[i])) for i, l in enumerate(labels)}
    return out


def format_scores(sc, title=""):
    return (f"{title}macro-F1 (labels in gold) {sc['macro_f1_present']:.4f} [{sc['labels_present']} labels] | "
            f"macro-F1 (all labels) {sc['macro_f1']:.4f} | micro-F1 {sc['micro_f1']:.4f} (P {sc['micro_p']:.3f} R {sc['micro_r']:.3f}) | "
            f"subset acc {sc['subset_acc']:.4f} | labels/doc pred {sc['avg_pred_labels']:.2f} gold {sc['avg_gold_labels']:.2f} | "
            f"docs with no prediction {sc['empty_pred_docs']} | n {sc['n']}")


GRID = np.round(np.arange(0.02, 0.981, 0.01), 2)


def decide(P, thr, min_labels=0):
    Pb = P >= np.asarray(thr)
    if min_labels:
        top = np.argsort(-P, axis=1)[:, :int(min_labels)]
        rows = np.arange(len(P))[:, None]
        need = Pb.sum(1) < int(min_labels)
        Pb[rows[need], top[need]] = True
    return Pb


def tune_global(P, Y, metric="macro_f1_present", min_labels=0):
    best_t, best = 0.5, -1.0
    for t in GRID:
        s = scores(decide(P, t, min_labels), Y)[metric]
        if s > best + 1e-12:
            best_t, best = float(t), s
    return best_t, best


def tune_per_label(P, Y, fallback):
    Y = np.asarray(Y, bool)
    thr = np.full(P.shape[1], float(fallback))
    for j in range(P.shape[1]):
        if not Y[:, j].any():
            continue
        pb = P[:, [j]] >= GRID[None, :]
        tp = (pb & Y[:, [j]]).sum(0); fp = (pb & ~Y[:, [j]]).sum(0); fn = (~pb & Y[:, [j]]).sum(0)
        f = np.divide(2 * tp, 2 * tp + fp + fn, out=np.zeros(len(GRID)), where=(2 * tp + fp + fn) > 0)
        thr[j] = float(GRID[int(f.argmax())])
    return thr


def validate_prediction(pred_list, test_cases, valid_labels):
    assert isinstance(pred_list, list), "prediction.json must be a JSON list"
    ids = [str(r["id"]) for r in pred_list]
    assert len(ids) == len(set(ids)), "duplicate ids in prediction"
    want = {str(v["id"]) for v in test_cases.values()}
    assert set(ids) == want, f"ids differ from test.json: missing {len(want - set(ids))}, extra {len(set(ids) - want)}"
    ok = set(int(l) for l in valid_labels)
    for r in pred_list:
        assert set(r) == {"id", "pred"}, f"each entry must be exactly {{id, pred}}, got {sorted(r)}"
        p = r["pred"]
        assert isinstance(p, list) and all(type(x) is int for x in p), f"pred must be a list of ints: {p!r}"
        assert len(p) == len(set(p)), f"duplicate statute ids in {r['id']}"
        assert set(p) <= ok, f"unknown statute ids {sorted(set(p) - ok)[:5]} in {r['id']}"
    return len(pred_list)


def dataset_stats(cases, toks):
    n = sorted(len(toks[k]) for k in cases)
    q = lambda f: n[min(len(n) - 1, int(len(n) * f))] if n else 0
    labs = [v["labels"] for v in cases.values() if v["labels"] is not None]
    cnt = Counter(l for ls in labs for l in ls)
    per = Counter(len(ls) for ls in labs)
    freq = sorted(cnt.values())
    return dict(cases=len(cases), tokens_p50=q(.5), tokens_p90=q(.9), tokens_max=n[-1] if n else 0,
                labelled=len(labs), distinct_labels=len(cnt),
                labels_per_doc={str(k): v for k, v in sorted(per.items())[:12]},
                label_freq_min=freq[0] if freq else 0, label_freq_p50=freq[len(freq) // 2] if freq else 0,
                label_freq_max=freq[-1] if freq else 0, labels_under_10_docs=sum(1 for c in freq if c < 10))

Writing lsi_data.py


In [5]:
%%writefile test_lsi_data.py
"""Synthetic tests for lsi_data.py (python test_lsi_data.py)."""
import json, tempfile

import numpy as np

import lsi_data as ld

ok = True


def check(name, cond, detail=""):
    global ok
    ok &= bool(cond)
    print(f"[{'PASS' if cond else 'FAIL'}] {name} {detail}")


for n in [0, 1, 513, 5000]:
    sp = ld.chunk_doc(n, 512, 412, None, "all")
    cov = set()
    for a, b in sp:
        cov.update(range(a, b))
    check(f"1 n={n}: whole doc covered", n == 0 or cov == set(range(n)))
check("1 head_tail", ld.chunk_doc(10000, 512, 512, 3, "head_tail") == ld.chunk_doc(10000, 512, 512, None, "all")[:2] + ld.chunk_doc(10000, 512, 512, None, "all")[-1:])


Y = np.array([[1, 0, 1], [0, 1, 0], [1, 0, 0]], bool)
P = np.array([[1, 0, 0], [0, 1, 1], [1, 0, 0]], bool)
sc = ld.scores(P, Y)

check("2 macro-F1", abs(sc["macro_f1"] - 2 / 3) < 1e-9 and abs(sc["macro_f1_present"] - 2 / 3) < 1e-9)
check("2 micro-F1 + subset acc", abs(sc["micro_f1"] - 2 * 3 / (2 * 3 + 1 + 1)) < 1e-9 and abs(sc["subset_acc"] - 1 / 3) < 1e-9)
Y2 = np.array([[1, 0], [1, 0]], bool); P2 = np.array([[1, 0], [1, 0]], bool)
s2 = ld.scores(P2, Y2)
check("2 labels without gold excluded from 'present' macro", s2["macro_f1_present"] == 1.0 and s2["macro_f1"] == 0.5)


rng = np.random.default_rng(0)
Yr = rng.random((200, 5)) < 0.2
Pr = np.clip(Yr * 0.3 + rng.random((200, 5)) * 0.6, 0, 1)
t, f = ld.tune_global(Pr, Yr)
check("3 global threshold beats 0.5", f >= ld.scores(ld.decide(Pr, 0.5), Yr)["macro_f1_present"], f"(t={t}, f={f:.3f})")
thr = ld.tune_per_label(Pr, Yr, t)
check("3 per-label >= global on dev", ld.scores(ld.decide(Pr, thr), Yr)["macro_f1_present"] >= f - 1e-9)
pb = ld.decide(np.array([[0.1, 0.2, 0.05]]), 0.9, min_labels=1)
check("3 min_labels forces top label", pb.tolist() == [[False, True, False]])


tmp = tempfile.mkdtemp()
rows = [dict(id=f"c{i}", text="x " * i, labels=[i % 3, (i + 1) % 3]) for i in range(6)]
json.dump(rows + [rows[0]], open(f"{tmp}/a.json", "w"))
a = ld.load_cases(f"{tmp}/a.json")
check("4 labels sorted unique ints, identical repeat dropped", len(a) == 6 and a["c1"]["labels"] == [1, 2])
json.dump([dict(id="t1", text="a"), dict(id="t2", text="b")], open(f"{tmp}/t.json", "w"))
t = ld.load_cases(f"{tmp}/t.json")
check("4 test without labels", all(v["labels"] is None for v in t.values()))
json.dump([dict(id="z", text="a", labels="7")], open(f"{tmp}/s.json", "w"))
check("4 scalar/str label -> [7]", ld.load_cases(f"{tmp}/s.json")["z"]["labels"] == [7])


good = [dict(id="t1", pred=[1, 7]), dict(id="t2", pred=[])]
check("5 accepts (empty list allowed)", ld.validate_prediction(good, t, [1, 7, 9]) == 2)
for bad, why in [([dict(id="t1", pred=[1, 1]), dict(id="t2", pred=[])], "duplicate ids in pred"),
                 ([dict(id="t1", pred=[3]), dict(id="t2", pred=[])], "unknown label"),
                 ([dict(id="t1", pred=[1.0]), dict(id="t2", pred=[])], "float label"),
                 ([dict(id="t1", pred=[1])], "missing doc")]:
    try:
        ld.validate_prediction(bad, t, [1, 7, 9]); check(f"5 rejects {why}", False)
    except AssertionError:
        check(f"5 rejects {why}", True)

print("\nALL TESTS PASSED" if ok else "\nSOME TESTS FAILED")

Writing test_lsi_data.py


In [6]:
%%writefile lsi_train.py
"""lsi_train.py — Phase-5 Legal Statute Identification harness."""
import importlib, json, math, os, random, sys, time, zipfile
from collections import Counter

import numpy as np
import torch
import torch.nn.functional as F

import lsi_data as ld

CFG = json.load(open(sys.argv[1]))
D, TR, MODEL_CFG = CFG["data"], CFG["train"], CFG["model"]
KD = CFG.get("kd") or {}
OUT, NAME = CFG["out_dir"], CFG["run_name"]
os.makedirs(OUT, exist_ok=True)
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = int(TR.get("seed", 0))
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)


def log(*a):
    print(*a, flush=True)


# ------------------------------------------------------------------ data
from tokenizers import ByteLevelBPETokenizer
TOK = ByteLevelBPETokenizer(vocab=os.path.join(D["tokenizer_dir"], "vocab.json"),
                            merges=os.path.join(D["tokenizer_dir"], "merges.txt"))
PAD_ID = TOK.token_to_id("<pad>") or 0
L, STRIDE = int(D["chunk_len"]), int(D.get("stride") or D["chunk_len"])
MAXC, SEL = D.get("max_chunks"), D.get("select", "first")
EMAXC, ESEL = D.get("eval_max_chunks", MAXC), D.get("eval_select", SEL)
assert MAXC and EMAXC, "set max_chunks and eval_max_chunks to numbers (very long documents would not fit in memory)"


def enc_batch(texts):
    return [e.ids for e in TOK.encode_batch(texts)]


PARTS, TOKS = {}, {}
RAW = {split: ld.load_cases(D[split]) for split in ("train", "dev", "test") if D.get(split)}
LC = D.get("lowercase", False)
if LC == "auto":
    LC = ld.auto_lowercase(RAW["train"], RAW.get("test", RAW.get("dev", {})))
log(f"lowercase: {LC} (setting {D.get('lowercase', False)!r}; all-lowercase docs: "
    + ", ".join(f"{s} {ld.lowercase_share(r):.0%}" for s, r in RAW.items()) + ")")
for split, raw in RAW.items():
    if LC:
        for v in raw.values():
            v["text"] = v["text"].lower()
    t = ld.tokenize_cases(raw, enc_batch, D.get("cache_dir", "lsi_cache"), tag=split + ("-lc" if LC else ""))
    PARTS[split] = {f"{split}::{k}": v for k, v in raw.items()}
    TOKS.update({f"{split}::{k}": v for k, v in t.items()})
TRAIN, DEVS, TEST = PARTS.get("train", {}), PARTS.get("dev", {}), PARTS.get("test", {})
assert TRAIN and all(v["labels"] is not None for v in TRAIN.values()), "train missing or has cases without labels"
LABELS = sorted({l for part in (TRAIN, DEVS) for v in part.values() for l in (v["labels"] or [])})
L2I = {l: i for i, l in enumerate(LABELS)}
C = len(LABELS)
if D.get("train_on_dev"):
    TRAIN = dict(TRAIN, **DEVS)
    DEVS = {}
if TR.get("train_subset"):
    _ks = random.Random(SEED).sample(sorted(TRAIN), min(len(TRAIN), int(TR["train_subset"])))
    TRAIN = {k: TRAIN[k] for k in _ks}


def multihot(keys, part):
    Y = np.zeros((len(keys), C), np.float32)
    for i, k in enumerate(keys):
        for l in part[k]["labels"] or []:
            if l in L2I:
                Y[i, L2I[l]] = 1.0
    return Y


def spans(key, train):
    return ld.chunk_doc(len(TOKS[key]), L, STRIDE, MAXC if train else EMAXC, SEL if train else ESEL)


TEACH = None
if KD.get("teacher_logits"):
    z = np.load(KD["teacher_logits"])
    meta = json.loads(str(z["__meta__"]))
    assert meta["labels"] == LABELS, "teacher used a different statute vocabulary"
    TEACH = {k[2:]: z[k].astype(np.float32) for k in z.files if k.startswith("d:")}
    covered = {k: v for k, v in TRAIN.items() if k in TEACH}
    assert covered, "teacher logits cover none of the train cases"
    if len(covered) < len(TRAIN):
        log(f"KD: teacher covers {len(covered)} of {len(TRAIN)} train cases -> the student trains on those {len(covered)}")
        TRAIN = covered
    log(f"KD: teacher logits for {len(TEACH)} cases | T={KD['temperature']} alpha={KD['alpha']}")

pos = multihot(list(TRAIN), TRAIN).sum(0)
log(f"train {len(TRAIN)} | dev {len(DEVS)} | test {len(TEST)} | statutes {C} | positives per statute: min {int(pos.min())} "
    f"median {int(np.median(pos))} max {int(pos.max())} | statutes with 0 train positives {int((pos == 0).sum())} | "
    f"chunk_len {L} stride {STRIDE} | train max_chunks {MAXC} {SEL} | eval max_chunks {EMAXC} {ESEL}")


def collate(keys, part, train):
    rows, did, cpos, crel = [], [], [], []
    for j, k in enumerate(keys):
        ids = TOKS[k]
        n = max(1, len(ids))
        for p, (a, b) in enumerate(spans(k, train)):
            seg = ids[a:b] if len(ids) else np.asarray([PAD_ID], np.uint16)
            rows.append(seg); did.append(j); cpos.append(p); crel.append(a / n)
    T = max(len(r) for r in rows)
    x = torch.full((len(rows), T), PAD_ID, dtype=torch.long)
    m = torch.zeros((len(rows), T), dtype=torch.bool)
    for i, r in enumerate(rows):
        x[i, :len(r)] = torch.from_numpy(r.astype(np.int64)); m[i, :len(r)] = True
    y = torch.from_numpy(multihot(keys, part)) if all(part[k]["labels"] is not None for k in keys) else None
    tch = torch.tensor(np.stack([TEACH[k] for k in keys])) if (train and TEACH is not None) else None
    return (x, m, torch.tensor(did), len(keys), torch.tensor(cpos), torch.tensor(crel, dtype=torch.float32)), y, tch


# ------------------------------------------------------------------ model
model_lib = importlib.import_module(CFG.get("model_module", "cj_model"))
build = getattr(model_lib, "build_lsi_model", None) or getattr(model_lib, "build_cj_model")
model = build(dict(MODEL_CFG), C).to(DEV)
n_total = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
log(f"parameters: total {n_total/1e6:.2f}M | trainable {n_train/1e6:.3f}M ({100*n_train/max(1,n_total):.2f}%)")
assert n_train > 0, "no trainable parameters"

if CFG.get("load_trained"):
    sd = torch.load(CFG["load_trained"], map_location="cpu", weights_only=False)["trained"]
    cur = dict(model.named_parameters())
    miss = [k for k in sd if k not in cur]
    assert not miss, f"load_trained: names not in this model, e.g. {miss[:3]}"
    with torch.no_grad():
        for k, v in sd.items():
            cur[k].copy_(v.to(cur[k].device, cur[k].dtype))
    log(f"loaded {len(sd)} trained tensors from {CFG['load_trained']}")


def run_model(inp):
    x, m, did, nd, cpos, crel = inp
    with torch.autocast(DEV.type, dtype=torch.float16, enabled=DEV.type == "cuda"):
        lg = model(x.to(DEV), m.to(DEV), did.to(DEV), nd, chunk_pos=cpos.to(DEV), chunk_rel=crel.to(DEV))
    assert tuple(lg.shape) == (nd, C), f"model returned {tuple(lg.shape)}, expected [{nd}, {C}]"
    return lg.float()


@torch.no_grad()
def predict_logits(part):
    model.eval()
    keys = sorted(part, key=lambda k: -len(spans(k, False)))
    budget = int(TR.get("eval_chunk_budget", max(1, int(TR["batch_size"])) * int(EMAXC)))
    out, i = {}, 0
    while i < len(keys):
        b, used = [], 0
        while i < len(keys) and (not b or used + len(spans(keys[i], False)) <= budget):
            used += len(spans(keys[i], False)); b.append(keys[i]); i += 1
        inp, _, _ = collate(b, part, train=False)
        for k, v in zip(b, run_model(inp).cpu().numpy()):
            out[k] = v
    model.train()
    return out


def probs(lg, keys):
    return 1 / (1 + np.exp(-np.stack([lg[k] for k in keys])))


DEV_KEYS = list(DEVS)
Y_DEV = multihot(DEV_KEYS, DEVS).astype(bool) if DEVS else None
MINL = int(TR.get("min_labels", 0))

# ------------------------------------------------------------------ training
bs, acc = int(TR["batch_size"]), int(TR.get("grad_accum", 1))
epochs = int(TR["epochs"])
keys_all = list(TRAIN)
per_epoch = min(len(keys_all), int(TR.get("epoch_cases") or len(keys_all)))
steps_per_epoch = math.ceil(per_epoch / (bs * acc))
total_steps = steps_per_epoch * epochs
warm = int(float(TR.get("warmup_frac", 0)) * total_steps)
base_lr = float(TR["lr"])
groups = model.param_groups(base_lr) if hasattr(model, "param_groups") else \
    [{"params": [p for p in model.parameters() if p.requires_grad]}]
opt = torch.optim.AdamW(groups, lr=base_lr, weight_decay=float(TR.get("weight_decay", 0.0)))
for g in opt.param_groups:
    g.setdefault("initial_lr", g["lr"])
scaler = torch.amp.GradScaler("cuda", enabled=DEV.type == "cuda")

pw = TR.get("pos_weight")
if pw == "balanced":
    neg = len(keys_all) - pos
    pw = np.clip(neg / np.maximum(pos, 1), 1.0, float(TR.get("pos_weight_max", 100.0))).tolist()
elif isinstance(pw, (int, float)):
    pw = [float(pw)] * C
pw_t = torch.tensor(pw, dtype=torch.float32, device=DEV) if pw is not None else None
LOSS = TR.get("loss", "bce")
GAMMA = float(TR.get("focal_gamma", 2.0))
log(f"loss {LOSS} | pos_weight {'none' if pw is None else f'min {min(pw):.2f} max {max(pw):.2f}'} | cases/epoch {per_epoch} | "
    f"steps/epoch {steps_per_epoch} | epochs {epochs}")


def task_loss(lg, y):
    if LOSS == "focal":
        bce = F.binary_cross_entropy_with_logits(lg, y, pos_weight=pw_t, reduction="none")
        p = torch.sigmoid(lg)
        pt = p * y + (1 - p) * (1 - y)
        return ((1 - pt) ** GAMMA * bce).mean()
    return F.binary_cross_entropy_with_logits(lg, y, pos_weight=pw_t)


def lr_mult(step):
    if step < warm:
        return (step + 1) / max(1, warm)
    t = (step - warm) / max(1, total_steps - warm)
    sch = TR.get("schedule", "linear")
    return 1.0 if sch == "constant" else max(0.0, 1 - t) if sch == "linear" else 0.5 * (1 + math.cos(math.pi * t))


SELECT = {"macro_f1": "dev_macro_f1", "tuned_macro_f1": "dev_tuned_macro_f1", "micro_f1": "dev_micro_f1"}[TR.get("select_metric", "macro_f1")]
best, best_state, history = -1.0, None, []
if epochs == 0:
    best_state = {k: v.detach().cpu().clone() for k, v in model.named_parameters() if v.requires_grad}
step, t0 = 0, time.time()
max_min = float(TR.get("max_minutes", 1e9))
if DEV.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
order = []
model.train()
for ep in range(1, epochs + 1):
    if len(order) < per_epoch:
        rest = keys_all[:]; random.shuffle(rest); order += rest
    ep_keys, order = order[:per_epoch], order[per_epoch:]
    run_loss, nb = 0.0, 0
    for i in range(0, len(ep_keys), bs):
        inp, y, tch = collate(ep_keys[i:i + bs], TRAIN, train=True)
        lg = run_model(inp)
        y = y.to(DEV)
        loss = task_loss(lg, y)
        if tch is not None:
            Tk, a = float(KD["temperature"]), float(KD["alpha"])
            soft = torch.sigmoid(tch.to(DEV) / Tk)
            kd = F.binary_cross_entropy_with_logits(lg / Tk, soft)
            loss = (1 - a) * loss + a * Tk * Tk * kd
        scaler.scale(loss / acc).backward()
        nb += 1; run_loss += loss.item()
        if nb % acc == 0:
            for g in opt.param_groups:
                g["lr"] = g["initial_lr"] * lr_mult(step)
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], float(TR.get("grad_clip", 1.0)))
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1
        if (time.time() - t0) / 60 > max_min:
            break
    rec = dict(epoch=ep, train_loss=run_loss / max(1, nb), minutes=(time.time() - t0) / 60)
    if DEVS:
        Pd = probs(predict_logits(DEVS), DEV_KEYS)
        s5 = ld.scores(ld.decide(Pd, 0.5, MINL), Y_DEV)
        tt, tf = ld.tune_global(Pd, Y_DEV, min_labels=MINL)
        rec.update(dev_macro_f1=s5["macro_f1_present"], dev_micro_f1=s5["micro_f1"], dev_tuned_macro_f1=tf,
                   dev_tuned_t=tt, dev_labels_per_doc=s5["avg_pred_labels"])
        score = rec[SELECT]
    else:
        score = ep
    history.append(rec)
    log(" | ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in rec.items()))
    if score >= best:
        best = score
        best_state = {k: v.detach().cpu().clone() for k, v in model.named_parameters() if v.requires_grad}
        log(f"  best so far (epoch {ep})")
    if (time.time() - t0) / 60 > max_min:
        log(f"time limit {max_min} min reached"); break
train_min = (time.time() - t0) / 60
peak_gb = torch.cuda.max_memory_allocated() / 1e9 if DEV.type == "cuda" else 0.0

cur = dict(model.named_parameters())
with torch.no_grad():
    for k, v in best_state.items():
        cur[k].copy_(v.to(DEV))
torch.save(dict(trained=best_state, cfg=CFG, labels=LABELS), os.path.join(OUT, f"{NAME}_trained_params.pt"))

# ------------------------------------------------------------------ thresholds (fixed 0.5 | global | per_label), on dev
THR = float(TR.get("threshold_value", 0.5))
mode = TR.get("threshold", "fixed")
result = dict(run=NAME, statutes=C, chunk_len=L, stride=STRIDE, max_chunks=MAXC, select=SEL, eval_max_chunks=EMAXC,
              eval_select=ESEL, params_total=n_total, params_trainable=n_train, train_minutes=train_min, peak_mem_gb=peak_gb,
              lowercase=bool(LC), loss=LOSS, pos_weight="none" if pw is None else TR.get("pos_weight"), min_labels=MINL, history=history)
if DEVS:
    t1 = time.time()
    dlg = predict_logits(DEVS)
    result["dev_infer_seconds"] = time.time() - t1
    Pd = probs(dlg, DEV_KEYS)
    s_fixed = ld.scores(ld.decide(Pd, THR, MINL), Y_DEV, LABELS)
    log(ld.format_scores(s_fixed, f"BEST on dev (threshold {THR}) "))
    result["dev_fixed"] = s_fixed
    if mode in ("global", "per_label"):
        tg, _ = ld.tune_global(Pd, Y_DEV, min_labels=MINL)
        THR = tg
        sg = ld.scores(ld.decide(Pd, THR, MINL), Y_DEV, LABELS)
        log(ld.format_scores(sg, f"BEST on dev (global threshold {tg:.2f}, tuned on dev) "))
        result["dev_global"] = sg
        if mode == "per_label":
            THR = ld.tune_per_label(Pd, Y_DEV, tg)
            sp = ld.scores(ld.decide(Pd, THR, MINL), Y_DEV, LABELS)
            log(ld.format_scores(sp, "BEST on dev (per-statute thresholds, tuned on dev) "))
            result["dev_per_label"] = sp
    final = ld.scores(ld.decide(Pd, THR, MINL), Y_DEV, LABELS)
    result.update(dev=final, threshold_mode=mode,
                  thresholds=THR.tolist() if isinstance(THR, np.ndarray) else THR)
    worst = sorted(((v["f1"], l, v["support"]) for l, v in final["per_label"].items() if v["support"]), key=lambda t: t[0])[:8]
    log("  lowest-F1 statutes on dev (F1, id, support): " + ", ".join(f"({f:.2f}, {l}, {s})" for f, l, s in worst))
    Pb = ld.decide(Pd, THR, MINL)
    json.dump([dict(id=DEVS[k]["id"], pred=[LABELS[j] for j in np.flatnonzero(Pb[i])], gold=DEVS[k]["labels"])
               for i, k in enumerate(DEV_KEYS)], open(os.path.join(OUT, f"{NAME}_dev_prediction.json"), "w"))
    np.savez_compressed(os.path.join(OUT, f"{NAME}_dev_logits.npz"), __meta__=json.dumps(dict(labels=LABELS)),
                        **{f"d:{k}": v.astype(np.float16) for k, v in dlg.items()})

if CFG.get("dump_teacher_logits"):
    tr_keys = sorted(PARTS.get("train", {}))
    if CFG.get("teacher_cases"):
        tr_keys = random.Random(SEED).sample(tr_keys, min(len(tr_keys), int(CFG["teacher_cases"])))
    every = {k: PARTS["train"][k] for k in tr_keys}; every.update(PARTS.get("dev", {}))
    t1 = time.time()
    tl = predict_logits(every)
    np.savez_compressed(os.path.join(OUT, f"{NAME}_teacher_logits.npz"), __meta__=json.dumps(dict(labels=LABELS, run=NAME)),
                        **{f"d:{k}": v.astype(np.float32) for k, v in tl.items()})
    log(f"teacher logits written for {len(tl)} cases ({(time.time() - t1) / 60:.1f} min)")

if TEST:
    t1 = time.time()
    tlg = predict_logits(TEST)
    tkeys = list(TEST)
    Pt = probs(tlg, tkeys)
    Pb = ld.decide(Pt, THR, MINL)
    pred = [dict(id=TEST[k]["id"], pred=[int(LABELS[j]) for j in np.flatnonzero(Pb[i])]) for i, k in enumerate(tkeys)]
    n = ld.validate_prediction(pred, TEST, LABELS)
    pj = os.path.join(OUT, f"{NAME}_prediction.json")
    json.dump(pred, open(pj, "w", encoding="utf-8"), ensure_ascii=False)
    zp = os.path.join(OUT, f"{NAME}_submission.zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(pj, arcname="prediction.json")
    np.savez_compressed(os.path.join(OUT, f"{NAME}_test_logits.npz"), **{f"d:{k}": v.astype(np.float16) for k, v in tlg.items()})
    lpd = [len(r["pred"]) for r in pred]
    result.update(test_cases=n, test_infer_seconds=time.time() - t1, test_labels_per_doc=float(np.mean(lpd)),
                  test_empty_docs=int(sum(1 for x in lpd if x == 0)))
    log(f"test: {n} docs | statutes per doc {np.mean(lpd):.2f} | docs with none {sum(1 for x in lpd if x == 0)} | infer {time.time() - t1:.0f}s")
    log(f"wrote {zp}")
json.dump(result, open(os.path.join(OUT, f"{NAME}_result.json"), "w"), indent=1, default=str)
log(f"done: {NAME} | best dev {SELECT} {best:.4f} | train {train_min:.1f} min | peak {peak_gb:.1f} GB | "
    f"trainable {n_train/1e6:.3f}M of {n_total/1e6:.2f}M")

Writing lsi_train.py


In [7]:
%%writefile phase1_loader.py
"""phase1_loader.py — load a Phase-1 model with its trained weights."""
import importlib, json, os

import torch

REGISTRY = os.environ.get("PHASE1_REGISTRY", "phase1_registry.json")


def load_phase1(name, map_location="cpu"):
    reg = json.load(open(REGISTRY))
    assert name in reg, f"{name!r} not in {REGISTRY}: {sorted(reg)}"
    r = reg[name]
    mod = importlib.import_module(r["module"])
    cfg = dict(r["model_cfg"])
    model = mod.build_model(cfg)
    ck = torch.load(r["ckpt"], map_location=map_location, weights_only=False)
    sd = ck["model"] if isinstance(ck, dict) and "model" in ck else ck
    missing, unexpected = model.load_state_dict(sd, strict=False)
    assert not missing and not unexpected, f"{name}: missing {missing[:5]} unexpected {unexpected[:5]}"
    val = ck.get("val_loss") if isinstance(ck, dict) else None
    return model, dict(model_cfg=cfg, ckpt=r["ckpt"], module=r["module"], val_loss=r.get("val_loss", val))

Overwriting phase1_loader.py


In [8]:
%%writefile cj_dummy.py
"""Smoke-test model only (NOT a submission model): bag of embeddings per chunk, mean over a case's chunks."""
import torch
import torch.nn as nn


class Dummy(nn.Module):
    def __init__(self, vocab_size, num_labels):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, 32)
        self.out = nn.Linear(32, num_labels)

    def forward(self, input_ids, attention_mask, doc_ids, n_docs, chunk_pos=None, chunk_rel=None):
        m = attention_mask.float()[..., None]
        chunk = (self.emb(input_ids) * m).sum(1) / m.sum(1).clamp(min=1)
        doc = torch.zeros(n_docs, chunk.shape[1], device=chunk.device, dtype=chunk.dtype).index_add_(0, doc_ids, chunk)
        cnt = torch.zeros(n_docs, device=chunk.device, dtype=chunk.dtype).index_add_(0, doc_ids, torch.ones_like(doc_ids, dtype=chunk.dtype))
        return self.out(doc / cnt.clamp(min=1)[:, None])


def build_cj_model(model_cfg, num_labels):
    return Dummy(int(model_cfg["vocab_size"]), num_labels)

Writing cj_dummy.py


In [9]:
%%writefile p1_biscale.py
"""model.py for the Bi-Scale generation notebook."""
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================== copied from the Bi-Scale training notebook ==============================
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return x * rms * self.weight

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq=2048, theta=10000.0):
        super().__init__()
        inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)
        self._build_cache(max_seq)

    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, device=self.inv_freq.device).float()
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos()[None, None, :, :])
        self.register_buffer("sin_cached", emb.sin()[None, None, :, :])

    def forward(self, seq_len):
        if seq_len > self.cos_cached.shape[2]:
            self._build_cache(seq_len)
        return self.cos_cached[:, :, :seq_len, :], self.sin_cached[:, :, :seq_len, :]

def rotate_half(x):
    x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
    return torch.cat([-x2, x1], dim=-1)

def apply_rotary(x, cos, sin):
    return x * cos + rotate_half(x) * sin

class BiScaleGQA(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, dropout, ctx_len, rope_theta, is_local=False, window_size=256):
        super().__init__()
        self.n_heads    = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_groups   = n_heads // n_kv_heads
        self.d_head     = d_model // n_heads

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.o_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_drop = nn.Dropout(dropout)
        self.rope = RotaryEmbedding(self.d_head, max_seq=ctx_len, theta=rope_theta)

        causal = torch.tril(torch.ones(ctx_len, ctx_len))
        if is_local:
            causal = causal * torch.triu(torch.ones(ctx_len, ctx_len), diagonal=-window_size)
        self.register_buffer("mask", causal.view(1, 1, ctx_len, ctx_len))


        for proj in [self.q_proj, self.k_proj, self.v_proj, self.o_proj]:
            nn.init.normal_(proj.weight, mean=0.0, std=0.02)

    def forward(self, x):
        B, T, C = x.shape
        dh = self.d_head

        q = self.q_proj(x).view(B, T, self.n_heads, dh).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, dh).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, dh).transpose(1, 2)

        cos, sin = self.rope(T)
        q = apply_rotary(q, cos, sin)
        k = apply_rotary(k, cos, sin)

        k = k.repeat_interleave(self.n_groups, dim=1)
        v = v.repeat_interleave(self.n_groups, dim=1)

        scale = dh ** -0.5
        att = (q @ k.transpose(-2, -1)) * scale
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = self.attn_drop(F.softmax(att, dim=-1))

        out = (att @ v).transpose(1, 2).contiguous().view(B, T, -1)
        return self.o_proj(out)

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate = nn.Linear(d_model, d_ff, bias=False)
        self.up   = nn.Linear(d_model, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d_model, bias=False)
        for proj in [self.gate, self.up, self.down]:
            nn.init.normal_(proj.weight, mean=0.0, std=0.02)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

class LearnedGatedResidual(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.gate = nn.Linear(dim, dim, bias=True)

        nn.init.normal_(self.gate.weight, mean=0.0, std=0.01)
        nn.init.constant_(self.gate.bias, 2.0)

    def forward(self, x, sublayer):
        g = torch.sigmoid(self.gate(x))
        return x + g * sublayer

class BiScaleLegalBlock(nn.Module):
    def __init__(self, cfg, layer_idx):
        super().__init__()
        is_local = (layer_idx < (cfg["n_layers"] // 2))
        self.norm1 = RMSNorm(cfg["d_model"])
        self.attn  = BiScaleGQA(
            cfg["d_model"], cfg["n_heads"], cfg["n_kv_heads"],
            cfg["dropout"], cfg["ctx_len"], cfg["rope_theta"],
            is_local=is_local, window_size=cfg["local_window"]
        )
        self.res1 = LearnedGatedResidual(cfg["d_model"])
        self.norm2 = RMSNorm(cfg["d_model"])
        self.ffn   = SwiGLU(cfg["d_model"], cfg["d_ff"])
        self.res2  = LearnedGatedResidual(cfg["d_model"])
        self.drop  = nn.Dropout(cfg["dropout"])

    def forward(self, x):
        x = self.res1(x, self.drop(self.attn(self.norm1(x))))
        x = self.res2(x, self.drop(self.ffn(self.norm2(x))))
        return x

class BiScaleLegalLLaMA(nn.Module):
    def __init__(self, cfg, vocab_size):
        super().__init__()
        self.ctx     = cfg["ctx_len"]
        self.tok_emb = nn.Embedding(vocab_size, cfg["d_model"])
        self.drop    = nn.Dropout(cfg["dropout"])
        self.blocks  = nn.ModuleList([BiScaleLegalBlock(cfg, i) for i in range(cfg["n_layers"])])
        self.norm    = RMSNorm(cfg["d_model"])
        self.lm_head = nn.Linear(cfg["d_model"], vocab_size, bias=False)


        self.lm_head.weight = self.tok_emb.weight


        nn.init.normal_(self.tok_emb.weight, mean=0.0, std=0.02)

    def forward(self, ids, targets=None):
        B, T = ids.shape
        x = self.drop(self.tok_emb(ids))
        for block in self.blocks:
            x = block(x)
        logits = self.lm_head(self.norm(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss
# ======================================= end of copied code =======================================


_ORIGINAL_ATTN_FORWARD = BiScaleGQA.forward


def _sdpa_attn_forward(self, x):
    B, T, C = x.shape
    dh = self.d_head
    q = self.q_proj(x).view(B, T, self.n_heads, dh).transpose(1, 2)
    k = self.k_proj(x).view(B, T, self.n_kv_heads, dh).transpose(1, 2)
    v = self.v_proj(x).view(B, T, self.n_kv_heads, dh).transpose(1, 2)
    cos, sin = self.rope(T)
    q = apply_rotary(q, cos.to(q.dtype), sin.to(q.dtype))
    k = apply_rotary(k, cos.to(k.dtype), sin.to(k.dtype))
    k = k.repeat_interleave(self.n_groups, dim=1)
    v = v.repeat_interleave(self.n_groups, dim=1)
    y = F.scaled_dot_product_attention(q, k, v, attn_mask=self.mask[:, :, :T, :T].bool(),
                                       dropout_p=0.0, scale=dh ** -0.5)
    return self.o_proj(y.transpose(1, 2).contiguous().view(B, T, -1))


def use_sdpa(flag=True):
    BiScaleGQA.forward = _sdpa_attn_forward if flag else _ORIGINAL_ATTN_FORWARD


class BiScaleForHarness(BiScaleLegalLLaMA):
    def forward(self, input_ids, prefix_lens=None):
        return super().forward(input_ids)[0]


def build_model(model_cfg):
    use_sdpa(model_cfg.get("use_sdpa", True))
    return BiScaleForHarness(model_cfg, model_cfg["vocab_size"])

Writing p1_biscale.py


In [10]:
# 3 — data check: layout, sizes, statute counts, labels per doc, lengths, CASING per split; builds both token caches
!python test_lsi_data.py | tail -1
import lsi_data as ld, importlib; importlib.reload(ld)
from tokenizers import ByteLevelBPETokenizer
_tok = ByteLevelBPETokenizer(vocab="tokenizer/vocab.json", merges="tokenizer/merges.txt")
V = _tok.get_vocab_size()
enc_batch = lambda texts: [e.ids for e in _tok.encode_batch(texts)]
CASES = {}
for split in ("train", "dev", "test"):
    raw = json.load(open(P5[split]))
    ex = raw[0] if isinstance(raw, list) else next(iter(raw.items()))
    print(f"{split}.json layout: {type(raw).__name__} | keys {list(ex)[:6] if isinstance(ex, dict) else '-'} | example: {str(ex)[:160]} ...")
    t0 = time.time()
    CASES[split] = ld.load_cases(P5[split])
    toks = ld.tokenize_cases(CASES[split], enc_batch, "lsi_cache", tag=split)
    texts = [v["text"] for v in CASES[split].values()]
    up = sum(sum(c.isupper() for c in t[:5000]) for t in texts) / max(1, sum(sum(c.isalpha() for c in t[:5000]) for t in texts))
    low = sum(1 for t in texts if not any(c.isupper() for c in t[:5000])) / len(texts)
    print(f"  {split}: " + json.dumps(ld.dataset_stats(CASES[split], toks)) + f" | uppercase share {up:.3f} | all-lowercase docs {low:.1%} | {time.time()-t0:.0f}s")
print("repeated ids:", dict(ld.DUPS))
LABELS = sorted({l for s in ("train", "dev") for v in CASES[s].values() for l in (v["labels"] or [])})
dev_only = sorted({l for v in CASES["dev"].values() for l in v["labels"]} - {l for v in CASES["train"].values() for l in v["labels"]})
print(f"statutes (train+dev): {len(LABELS)} | ids {LABELS[:10]} ... {LABELS[-5:]} | in dev but never in train: {dev_only[:10]}")

ALL TESTS PASSED
train.json layout: list | keys ['id', 'text', 'labels'] | example: {'id': '1000008', 'text': '(a), <SECTION> of the <ACT>. The gist of theprosecution case relevant <ENTITY> the purpose of this proceedingmay be stated thus: With ...
  train: {"cases": 42750, "tokens_p50": 676, "tokens_p90": 4343, "tokens_max": 148490, "labelled": 42750, "distinct_labels": 100, "labels_per_doc": {"1": 9146, "2": 7586, "3": 7046, "4": 5781, "5": 4264, "6": 3284, "7": 2131, "8": 1390, "9": 836, "10": 458, "11": 358, "12": 166}, "label_freq_min": 272, "label_freq_p50": 684, "label_freq_max": 11730, "labels_under_10_docs": 0} | uppercase share 0.116 | all-lowercase docs 0.0% | 95s
dev.json layout: list | keys ['id', 'text', 'labels'] | example: {'id': '1000139', 'text': "Dr. <ENTITY>, J. The accused persons are the <ENTITY> andbrother respectively of the deceased. The incident purportedly took place on ...
  dev: {"cases": 10181, "tokens_p50": 855, "tokens_p90": 4676, "tokens_max": 122449, "

In [11]:
# 3b — lowercase token cache, built only if lowercase="auto" would switch it on (test lowercase, train not)
NEED_LC = ld.auto_lowercase(CASES["train"], CASES["test"])
print("auto lowercase would be:", NEED_LC)
if NEED_LC:
    for split in ("train", "dev", "test"):
        raw = ld.load_cases(P5[split])
        for v in raw.values(): v["text"] = v["text"].lower()
        t0 = time.time(); ld.tokenize_cases(raw, enc_batch, "lsi_cache", tag=split + "-lc")
        print(f"{split}: lowercase cache ready ({time.time()-t0:.0f}s)")

auto lowercase would be: False


In [12]:
# 4 — Phase-1 models -> phase1_registry.json (same as Phase 2)
REG = {}
V = _tok.get_vocab_size()
def near(ckpt, fn):
    root = os.path.dirname(os.path.dirname(ckpt)) if os.path.basename(os.path.dirname(ckpt)) == "run" else os.path.dirname(ckpt)
    hits = sorted(glob.glob(f"{root}/**/{fn}", recursive=True), key=len)
    return hits[0] if hits else None
for name in ["prefixlm", "encdec", "arch4"]:
    ck = find(f"{name}_best.pt")
    if not ck:
        print(f"{name}: NOT FOUND (attach phase1-models)"); continue
    shutil.copy(near(ck[0], "model.py"), f"p1_{name}.py")
    cfgj = json.load(open(near(ck[0], "run_config.json")))
    mc = dict(cfgj["model"]); mc.setdefault("vocab_size", V); mc.setdefault("max_seq_len", int(cfgj["train"]["ctx_len"]))
    REG[name] = dict(module=f"p1_{name}", model_cfg=mc, ckpt=ck[0])
bs = find("stage2_done.pt")
if bs:
    REG["biscale"] = dict(module="p1_biscale", ckpt=bs[0],
                          model_cfg=dict(n_layers=16, n_heads=12, n_kv_heads=4, d_model=768, d_ff=2048, ctx_len=1024,
                                         local_window=256, dropout=0.1, rope_theta=10000.0, vocab_size=V))
json.dump(REG, open("phase1_registry.json", "w"), indent=1)
from phase1_loader import load_phase1
for name in REG:
    m, info = load_phase1(name)
    print(f"{name:9s} {sum(p.numel() for p in m.parameters())/1e6:7.2f}M params | {info['ckpt']}")
    del m

prefixlm   109.63M params | /kaggle/input/datasets/karanchauhan435257/phase1-models/prefixlm-final/run/prefixlm_best.pt
encdec     114.37M params | /kaggle/input/datasets/karanchauhan435257/phase1-models/encdec-final/run/encdec_best.pt
arch4      109.60M params | /kaggle/input/datasets/karanchauhan435257/phase1-models/arch4-final/run/arch4_best.pt
biscale    144.16M params | /kaggle/input/datasets/karanchauhan435257/phase1-models/biscale-final/checkpoints/stage2_done.pt


In [13]:
# 5 — experiment config helper + runners
TODO = "TODO"
R = "/kaggle/working/p5runs"
def make_cfg(run_name, model, train, data=None, kd=None, dump_teacher=False, module="cj_model", teacher_cases=None):
    d = dict(tokenizer_dir="tokenizer", train=P5["train"], dev=P5["dev"], test=P5["test"], cache_dir="lsi_cache",
             chunk_len=TODO, stride=None, max_chunks=TODO, select=TODO)


    d.update(data or {})
    cfg = dict(run_name=run_name, out_dir=f"{R}/{run_name}", model_module=module, data=d, model=model, train=train,
               kd=kd, dump_teacher_logits=dump_teacher, teacher_cases=teacher_cases)
    todo = [k for k, v in {**d, **train, **(kd or {}), **model}.items() if v == TODO]
    assert not todo, f"{run_name}: fill {todo}"
    os.makedirs(cfg["out_dir"], exist_ok=True)
    p = f"{cfg['out_dir']}/config.json"; json.dump(cfg, open(p, "w"), indent=1)
    return p

def run_on_gpu(cp, gpu):
    lf = open(os.path.join(os.path.dirname(cp), "train.log"), "w")
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    subprocess.Popen([sys.executable, "lsi_train.py", cp], stdout=lf, stderr=subprocess.STDOUT, env=env, start_new_session=True)
    print("started", cp, "on GPU", gpu)

def run_parallel(cfg_paths, timeout_min=240):
    procs, queue = [], list(cfg_paths)
    while queue or procs:
        while queue and len(procs) < max(1, NGPU):
            cp = queue.pop(0); gpu = [g for g in range(max(1, NGPU)) if g not in [x[2] for x in procs]][0]
            lf = open(os.path.join(os.path.dirname(cp), "train.log"), "w")
            env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
            procs.append((subprocess.Popen([sys.executable, "lsi_train.py", cp], stdout=lf, stderr=subprocess.STDOUT, env=env,
                                           start_new_session=True), cp, gpu, lf, time.time()))
            print(f"started {cp} on GPU {gpu}")
        time.sleep(15)
        for pr in list(procs):
            p, cp, gpu, lf, t0 = pr
            if p.poll() is not None or time.time() - t0 > timeout_min * 60:
                if p.poll() is None: p.kill()
                lf.close(); procs.remove(pr)
                tail = open(os.path.join(os.path.dirname(cp), "train.log")).read().splitlines()[-25:]
                print(f"\n==== {cp} exit {p.returncode} ({(time.time()-t0)/60:.1f} min)\n" + "\n".join(tail))

def tail_log(name, n=8):
    print("\n".join(open(f"{R}/{name}/train.log").read().splitlines()[-n:]))

def teacher_from(run, new, teacher_cases=None):
    c = json.load(open(f"{R}/{run}/config.json"))
    p = make_cfg(new, model=c["model"], train=dict(c["train"], epochs=0), data=c["data"], module=c["model_module"],
                 dump_teacher=True, teacher_cases=teacher_cases)
    cfg = json.load(open(p)); cfg["load_trained"] = f"{R}/{run}/{run}_trained_params.pt"
    json.dump(cfg, open(p, "w"), indent=1); return p

def _cfg_for(job):
    if job.get("teacher_of"):
        return teacher_from(job["teacher_of"], job["name"], job.get("teacher_cases"))
    kd = None
    if job.get("kd"):
        k = dict(job["kd"]); t = k.pop("teacher"); k.pop("teacher_cases", None)
        kd = dict(teacher_logits=f"{R}/{t}_teacher/{t}_teacher_teacher_logits.npz", **k)
    return make_cfg(job["name"], job["model"], job["train"], job.get("data"), kd=kd, module=job.get("module", "cj_model"))

def run_queues(queues, poll=20):
    qs = {}
    for g, jobs in queues.items():
        lst = []
        for j in jobs:
            if j.get("kd"):
                t = j["kd"]["teacher"]
                lst.append(dict(name=f"{t}_teacher", teacher_of=t, teacher_cases=j["kd"].get("teacher_cases")))
            lst.append(j)
        qs[int(g)] = lst
    cur, t_start = {}, time.time()
    while any(qs.values()) or cur:
        for g in list(qs):
            if g not in cur and qs[g]:
                j = qs[g].pop(0)
                try:
                    cp = _cfg_for(j)
                except Exception as e:
                    print(f"SKIPPED {j['name']} on GPU {g}: {type(e).__name__}: {e}", flush=True); continue
                lf = open(os.path.join(os.path.dirname(cp), "train.log"), "w")
                env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
                p = subprocess.Popen([sys.executable, "lsi_train.py", cp], stdout=lf, stderr=subprocess.STDOUT, env=env,
                                     start_new_session=True)
                cur[g] = (p, j["name"], lf, time.time())
                print(f"[{(time.time()-t_start)/60:6.1f} min] started {j['name']} on GPU {g}", flush=True)
        time.sleep(poll)
        for g, (p, name, lf, t0) in list(cur.items()):
            if p.poll() is not None:
                lf.close(); del cur[g]
                tail = open(f"{R}/{name}/train.log").read().splitlines()[-14:]
                print(f"\n==== [{(time.time()-t_start)/60:6.1f} min] {name} exit {p.returncode} ({(time.time()-t0)/60:.1f} min)\n"
                      + "\n".join(tail) + "\n", flush=True)

def summary(names):
    for n in names:
        z, res = f"{R}/{n}/{n}_submission.zip", f"{R}/{n}/{n}_result.json"
        if os.path.exists(z) and os.path.exists(res):
            r = json.load(open(res)); dv = r.get("dev", {})
            print(f"OK  {n:24s} dev macro-F1 {dv.get('macro_f1_present', float('nan')):.4f} | micro-F1 {dv.get('micro_f1', float('nan')):.4f} | "
                  f"threshold {r.get('threshold_mode')} | test statutes/doc {r.get('test_labels_per_doc', float('nan')):.2f} | "
                  f"trainable {r['params_trainable']/1e6:.2f}M of {r['params_total']/1e6:.1f}M | {r['train_minutes']:.1f} min -> {z}")
        else:
            print(f"FAILED {n}: read {R}/{n}/train.log")

In [14]:
# 6 — smoke test: dummy model; fixed/global/per-label thresholds, focal + pos_weight, teacher -> KD student, zip
SM_T = dict(epochs=2, epoch_cases=256, lr=1e-3, batch_size=16, max_minutes=2)
SM_D = dict(chunk_len=512, max_chunks=2, select="head_tail")
a = make_cfg("smoke_teacher", dict(vocab_size=V), dict(SM_T, threshold="global"), SM_D, module="cj_dummy",
             dump_teacher=True, teacher_cases=500)
b = make_cfg("smoke_perlabel", dict(vocab_size=V), dict(SM_T, threshold="per_label", loss="focal", pos_weight="balanced", min_labels=1),
             dict(chunk_len=256, stride=192, max_chunks=3, select="last", eval_max_chunks=4), module="cj_dummy")
run_parallel([a, b])
c = make_cfg("smoke_kd", dict(vocab_size=V), SM_T, SM_D, module="cj_dummy",
             kd=dict(teacher_logits=f"{R}/smoke_teacher/smoke_teacher_teacher_logits.npz", temperature=2.0, alpha=0.5))
run_parallel([c])
summary(["smoke_teacher", "smoke_perlabel", "smoke_kd"])
assert all(os.path.exists(f"{R}/{n}/{n}_submission.zip") for n in ["smoke_teacher", "smoke_perlabel", "smoke_kd"]), "SMOKE TEST FAILED"
import zipfile; zf = zipfile.ZipFile(f"{R}/smoke_perlabel/smoke_perlabel_submission.zip")
print("zip contains:", zf.namelist(), "| first entries:", json.loads(zf.read("prediction.json"))[:2])
print("SMOKE TEST PASSED (pipeline only: the dummy model's scores mean nothing)")

started /kaggle/working/p5runs/smoke_teacher/config.json on GPU 0
started /kaggle/working/p5runs/smoke_perlabel/config.json on GPU 1

==== /kaggle/working/p5runs/smoke_teacher/config.json exit 0 (0.5 min)
lowercase: False (setting False; all-lowercase docs: train 0%, dev 0%, test 0%)
train 42750 | dev 10181 | test 13019 | statutes 100 | positives per statute: min 272 median 679 max 11730 | statutes with 0 train positives 0 | chunk_len 512 stride 512 | train max_chunks 2 head_tail | eval max_chunks 2 head_tail
parameters: total 1.03M | trainable 1.027M (100.00%)
loss bce | pos_weight none | cases/epoch 256 | steps/epoch 16 | epochs 2
epoch 1 | train_loss 0.6851 | minutes 0.0198 | dev_macro_f1 0.0444 | dev_micro_f1 0.0795 | dev_tuned_macro_f1 0.0715 | dev_tuned_t 0.4100 | dev_labels_per_doc 34.3532
  best so far (epoch 1)
epoch 2 | train_loss 0.6679 | minutes 0.0483 | dev_macro_f1 0.0403 | dev_micro_f1 0.0764 | dev_tuned_macro_f1 0.0715 | dev_tuned_t 0.4100 | dev_labels_per_doc 29.6842
B

In [15]:
%%writefile cj_model.py
"""cj_model.py — Phase-4/5 model."""
import math
import sys

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

from phase1_loader import load_phase1

BLOCKS = {"prefixlm": "blocks", "encdec": "encoder", "arch4": "blocks", "biscale": "blocks"}
FINAL_NORM = {"prefixlm": "ln_f", "encdec": "enc_ln", "arch4": "ln_f", "biscale": "norm"}
DEFAULT_TARGETS = {
    "prefixlm": ["attn.qkv", "attn.proj", "mlp.fc1", "mlp.fc2"],
    "encdec": ["attn.qkv", "attn.proj", "mlp.fc1", "mlp.fc2"],
    "arch4": ["msr.proj", "msr.out", "mlp.fc1", "mlp.fc2"],
    "biscale": ["attn.q_proj", "attn.k_proj", "attn.v_proj", "attn.o_proj", "ffn.gate", "ffn.up", "ffn.down"],
}
ADAPTER_SITES = {
    "prefixlm": ("attn", "mlp"), "encdec": ("attn", "mlp"),
    "arch4": ("msr", "mlp"), "biscale": ("attn", "ffn"),
}


def _first_attr(obj, names):
    for n in names:
        if getattr(obj, n, None) is not None:
            return getattr(obj, n)
    raise AttributeError(f"{type(obj).__name__} has none of {names}")


def _out_splits(backbone_name, path, lin, d_model):
    if path.endswith("attn.qkv"):
        return [lin.out_features // 3] * 3
    if path.endswith("cross.kv"):
        return [lin.out_features // 2] * 2
    if backbone_name == "arch4" and path.endswith("msr.proj"):
        vd = (lin.out_features - 2 * d_model) // 2
        return [d_model, d_model, vd, vd]
    return [lin.out_features]


class LoRMALinear(nn.Module):
    def __init__(self, base, r, alpha, splits, variant):
        super().__init__()
        self.base, self.splits, self.variant = base, list(splits), variant
        self.scale = alpha / r
        self.A = nn.ParameterList()
        self.B = nn.ParameterList()
        for d in self.splits:
            if variant == "plus":
                A = torch.empty(r, d)
                nn.init.kaiming_uniform_(A, a=math.sqrt(5))
                B = torch.zeros(d, r)
            elif variant == "pi":
                A = torch.zeros(r, d)
                A[0, 0] = 1.0
                B = torch.randn(d, r) / math.sqrt(r)
                B[:, 0] = 1.0
            else:
                raise ValueError(f"lorma_variant must be 'plus' or 'pi', got {variant!r}")
            self.A.append(nn.Parameter(A))
            self.B.append(nn.Parameter(B))

    def forward(self, x):
        y = F.linear(x, self.base.weight)
        outs = []
        for i, part in enumerate(y.split(self.splits, dim=-1)):
            A, B = self.A[i], self.B[i]
            if self.variant == "plus":
                outs.append(part + F.linear(F.linear(part, A.to(part.dtype)), B.to(part.dtype)) * self.scale)
            else:
                M = B @ A
                d = M.shape[0]
                idx = torch.arange(d, device=M.device)
                R = torch.gather(M, 1, (idx[None, :] - idx[:, None]) % d)
                outs.append(F.linear(part, R.to(part.dtype)))
        y = torch.cat(outs, dim=-1) if len(outs) > 1 else outs[0]
        if self.base.bias is not None:
            y = y + self.base.bias.to(y.dtype)
        return y


class Adapter(nn.Module):
    def __init__(self, d, m):
        super().__init__()
        self.down = nn.Linear(d, m)
        self.up = nn.Linear(m, d)
        nn.init.normal_(self.down.weight, 0.0, 0.02)
        nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, h):
        return h + self.up(F.gelu(self.down(h)))


class WithAdapter(nn.Module):
    def __init__(self, inner, d, m):
        super().__init__()
        self.inner = inner
        self.adapter = Adapter(d, m)

    def forward(self, *args, **kwargs):
        out = self.inner(*args, **kwargs)
        if isinstance(out, tuple):
            return (self.adapter(out[0]),) + tuple(out[1:])
        return self.adapter(out)

    def __getattr__(self, name):
        try:
            return super().__getattr__(name)
        except AttributeError:
            return getattr(self._modules["inner"], name)


class AttnContext:
    def __init__(self):
        self.key_mask = None
        self.capture = False


class PrefixLMAttention(nn.Module):
    def __init__(self, orig, ctx, rope_fn):
        super().__init__()
        self.orig, self.ctx, self.rope_fn = orig, ctx, rope_fn
        self.pk = self.pv = None

    def add_prefix(self, n):
        o = self.orig
        self.pk = nn.Parameter(torch.zeros(o.h, n, o.hd))
        self.pv = nn.Parameter(torch.zeros(o.h, n, o.hd))

    def forward(self, x, cos, sin, mask, kv=None, start=0):
        o = self.orig
        B, T, _ = x.shape
        q, k, v = o.qkv(x).view(B, T, 3, o.h, o.hd).permute(2, 0, 3, 1, 4)
        if o.rope:
            q, k = self.rope_fn(q, cos.to(q.dtype), sin.to(q.dtype)), self.rope_fn(k, cos.to(q.dtype), sin.to(q.dtype))
        if self.pk is not None and self.ctx.capture:
            self.pk.data.copy_(k[0].detach().float())
            self.pv.data.copy_(v[0].detach().float())
        elif self.pk is not None:
            n = self.pk.shape[1]
            k = torch.cat([self.pk.to(k.dtype)[None].expand(B, -1, -1, -1), k], dim=2)
            v = torch.cat([self.pv.to(v.dtype)[None].expand(B, -1, -1, -1), v], dim=2)
            mask = torch.cat([mask.new_ones(mask.shape[0], 1, T, n), mask.expand(-1, -1, T, -1)], dim=-1)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask,
                                           dropout_p=o.p_drop if self.training else 0.0)
        return o.proj(y.transpose(1, 2).reshape(B, T, o.d))


class EncoderAttention(nn.Module):
    def __init__(self, orig, ctx, rope_fn):
        super().__init__()
        self.orig, self.ctx, self.rope_fn = orig, ctx, rope_fn
        self.pk = self.pv = None

    def add_prefix(self, n):
        o = self.orig
        self.pk = nn.Parameter(torch.zeros(o.h, n, o.hd))
        self.pv = nn.Parameter(torch.zeros(o.h, n, o.hd))

    def forward(self, x, cos, sin, causal=False, cache=None, li=None, start=0):
        o = self.orig
        B, T, _ = x.shape
        q, k, v = o.qkv(x).view(B, T, 3, o.h, o.hd).permute(2, 0, 3, 1, 4)
        q, k = self.rope_fn(q, cos, sin), self.rope_fn(k, cos, sin)
        km = self.ctx.key_mask
        if self.pk is not None and self.ctx.capture:
            self.pk.data.copy_(k[0].detach().float())
            self.pv.data.copy_(v[0].detach().float())
        elif self.pk is not None:
            n = self.pk.shape[1]
            k = torch.cat([self.pk.to(k.dtype)[None].expand(B, -1, -1, -1), k], dim=2)
            v = torch.cat([self.pv.to(v.dtype)[None].expand(B, -1, -1, -1), v], dim=2)
            if km is not None:
                km = torch.cat([km.new_ones(B, n), km], dim=1)
        mask = None if km is None else km[:, None, None, :]
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask,
                                           dropout_p=o.p if self.training else 0.0)
        return o.proj(y.transpose(1, 2).reshape(B, T, o.d))


class CaseHead(nn.Module):
    def __init__(self, d, num_labels, pool, agg, hidden, dropout, use_position, out_prior=None):
        super().__init__()
        if pool not in ("mean", "last", "first", "attn", "mean_last"):
            raise ValueError(f"pool must be mean | last | first | attn | mean_last, got {pool!r}")
        if agg not in ("mean", "max", "attn", "gru", "chunk_max", "chunk_lme"):
            raise ValueError(f"agg must be mean | max | attn | gru | chunk_max | chunk_lme, got {agg!r}")
        self.pool, self.agg = pool, agg
        self.tok_scorer = nn.Linear(d, 1) if pool == "attn" else None
        d_in = 2 * d if pool == "mean_last" else d
        self.drop = nn.Dropout(dropout)
        self.proj = nn.Sequential(nn.Linear(d_in, hidden), nn.GELU())
        self.pos = None
        if use_position:
            self.pos = nn.Linear(2, hidden)
            nn.init.zeros_(self.pos.weight)
            nn.init.zeros_(self.pos.bias)
        self.rnn = nn.GRU(hidden, hidden // 2, batch_first=True, bidirectional=True) if agg == "gru" else None
        self.chunk_scorer = nn.Linear(hidden, 1) if agg in ("attn", "gru") else None
        self.out = nn.Linear(hidden, num_labels)
        nn.init.normal_(self.out.weight, 0.0, 0.02)
        nn.init.zeros_(self.out.bias)
        if out_prior is not None:
            p = torch.as_tensor(out_prior, dtype=torch.float32).clamp(1e-4, 1 - 1e-4).expand(num_labels)
            with torch.no_grad():
                self.out.bias.copy_(torch.log(p / (1 - p)))

    def pool_tokens(self, h, mask):
        N, T, d = h.shape
        h = h.float()
        m = mask.float()
        cnt = m.sum(1).clamp(min=1.0)
        vecs = []
        if self.pool in ("mean", "mean_last"):
            vecs.append((h * m[..., None]).sum(1) / cnt[:, None])
        if self.pool in ("last", "mean_last"):
            pos = torch.arange(T, device=h.device, dtype=torch.float32)[None, :]
            idx = (pos * m).amax(1).long()
            vecs.append(h[torch.arange(N, device=h.device), idx])
        if self.pool == "first":
            pos = torch.arange(T, device=h.device, dtype=torch.float32)[None, :]
            idx = torch.where(m > 0, pos, torch.full_like(pos, float(T))).amin(1).clamp(max=T - 1).long()
            vecs.append(h[torch.arange(N, device=h.device), idx])
        if self.pool == "attn":
            score = self.tok_scorer(h).squeeze(-1).masked_fill(~mask, -1e4)
            w = torch.softmax(score, dim=-1)
            vecs.append(torch.einsum("nt,ntd->nd", w, h))
        return torch.cat(vecs, dim=-1)

    def forward(self, h, mask, doc_ids, n_docs, chunk_pos, chunk_rel):
        with torch.autocast(device_type=h.device.type, enabled=False):
            v = self.pool_tokens(h, mask)
            z = self.proj(self.drop(v))
            N = z.shape[0]
            counts = torch.bincount(doc_ids, minlength=n_docs)
            if self.pos is not None:
                rel = chunk_rel.float() if chunk_rel is not None else torch.zeros(N, device=z.device)
                first = torch.cumsum(counts, 0) - counts
                slot_f = (torch.arange(N, device=z.device) - first[doc_ids]).float()
                frac = slot_f / (counts[doc_ids].float() - 1.0).clamp(min=1.0)
                z = z + self.pos(torch.stack([rel, frac], dim=-1))
            if self.agg in ("chunk_max", "chunk_lme"):
                logits = self.out(self.drop(z))
                member = doc_ids[None, :] == torch.arange(n_docs, device=z.device)[:, None]
                masked = torch.where(member[..., None], logits[None], torch.full_like(logits[None], -1e4))
                if self.agg == "chunk_max":
                    return masked.amax(1)
                return torch.logsumexp(masked, dim=1) - torch.log(counts.clamp(min=1).float())[:, None]
            if self.agg == "mean":
                case = torch.zeros(n_docs, z.shape[1], device=z.device).index_add_(0, doc_ids, z)
                case = case / counts.clamp(min=1)[:, None].float()
            elif self.agg == "max":
                member = doc_ids[None, :] == torch.arange(n_docs, device=z.device)[:, None]
                case = torch.where(member[..., None], z[None], torch.full_like(z[None], -1e4)).amax(1)
            else:
                first = torch.cumsum(counts, 0) - counts
                slot = torch.arange(N, device=z.device) - first[doc_ids]
                M = int(counts.max().item())
                seq = z.new_zeros(n_docs, M, z.shape[1])
                seq[doc_ids, slot] = z
                valid = torch.zeros(n_docs, M, dtype=torch.bool, device=z.device)
                valid[doc_ids, slot] = True
                if self.rnn is not None:
                    packed = nn.utils.rnn.pack_padded_sequence(seq, counts.clamp(min=1).cpu(), batch_first=True,
                                                               enforce_sorted=False)
                    out, _ = self.rnn(packed)
                    seq, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=M)
                score = self.chunk_scorer(seq).squeeze(-1).masked_fill(~valid, -1e4)
                w = torch.softmax(score, dim=-1) * valid.float()
                self.last_chunk_weights = w.detach()
                case = torch.einsum("bm,bmd->bd", w, seq)
            return self.out(self.drop(case))


class CJModel(nn.Module):
    def __init__(self, cfg, num_labels):
        super().__init__()
        self.cfg = cfg
        self.name = cfg.get("phase1", "prefixlm")
        if self.name not in BLOCKS:
            raise ValueError(f"phase1 must be one of {list(BLOCKS)}, got {self.name!r}")
        self.method = cfg.get("method", "full")
        backbone, self.info = load_phase1(self.name)
        self.backbone = backbone
        self.p1 = sys.modules[type(backbone).__module__]
        self.ctx = AttnContext()
        self.use_decoder = self.name == "encdec" and bool(cfg.get("encdec_decoder", False))
        self.grad_ckpt = bool(cfg.get("grad_ckpt", False))

        self.emb_name = next(n for n in ("tok_emb", "embed_tokens", "embed", "wte")
                             if isinstance(getattr(backbone, n, None), nn.Embedding))
        self.d = self.emb.embedding_dim
        self._strip_lm_head()
        self._make_student()
        if cfg.get("backbone_dropout") is not None:
            for mod in backbone.modules():
                if isinstance(mod, nn.Dropout):
                    mod.p = float(cfg["backbone_dropout"])
        if self.name == "encdec":
            self._wrap_encoder_attention()
        if self.name == "prefixlm":
            self._wrap_prefixlm_attention()

        self.n_prompt = 0
        self.prompt = None
        self._apply_method()

        default_pool = "mean_last" if self.name in ("arch4", "biscale") else "mean"
        self.head = CaseHead(self.d, num_labels, cfg.get("pool", default_pool), cfg.get("agg", "gru"),
                             int(cfg.get("head_hidden", 256)), float(cfg.get("head_dropout", 0.1)),
                             bool(cfg.get("chunk_position", True)), cfg.get("out_prior"))

    @property
    def emb(self):
        return getattr(self.backbone, self.emb_name)

    def _blocks(self):
        return getattr(self.backbone, BLOCKS[self.name])

    def _stacks(self):
        s = [self._blocks()]
        if self.use_decoder:
            s.append(self.backbone.decoder)
        return s

    def _strip_lm_head(self):
        bb = self.backbone
        V = self.emb.num_embeddings
        for n, mod in list(bb.named_children()):
            if isinstance(mod, nn.Linear) and mod.out_features == V:
                setattr(bb, n, None)
        if self.name == "encdec" and not self.use_decoder:
            bb.decoder = None
            bb.dec_ln = None

    def _make_student(self):
        k = self.cfg.get("keep_layers")
        if not k:
            return
        blocks = self._blocks()
        n = len(blocks)
        k = int(k)
        if self.cfg.get("keep_mode", "first") == "spread":
            idx = sorted({round(i * (n - 1) / max(1, k - 1)) for i in range(k)})
        else:
            idx = list(range(k))
        setattr(self.backbone, BLOCKS[self.name], nn.ModuleList([blocks[i] for i in idx]))
        self.kept_layers = idx

    def _wrap_encoder_attention(self):
        for blk in self.backbone.encoder:
            blk.attn = EncoderAttention(blk.attn, self.ctx, self.p1.apply_rope)

    def _wrap_prefixlm_attention(self):
        for blk in self.backbone.blocks:
            blk.attn = PrefixLMAttention(blk.attn, self.ctx, self.p1.apply_rope)

    def _apply_method(self):
        cfg, m = self.cfg, self.method
        bb = self.backbone
        if m == "full":
            for p in bb.parameters():
                p.requires_grad = True
            return
        for p in bb.parameters():
            p.requires_grad = False
        if m == "partial":
            n_top = int(cfg.get("unfreeze_layers", 4))
            for stack in self._stacks():
                for blk in list(stack)[-n_top:]:
                    for p in blk.parameters():
                        p.requires_grad = True
            for p in getattr(bb, FINAL_NORM[self.name]).parameters():
                p.requires_grad = True
            if self.use_decoder:
                for p in bb.dec_ln.parameters():
                    p.requires_grad = True
            if cfg.get("unfreeze_embeddings", False):
                self.emb.weight.requires_grad = True
        elif m == "bitfit":
            for n, p in bb.named_parameters():
                if n.endswith(".bias"):
                    p.requires_grad = True
        elif m == "lorma":
            self._wrap_linears(m)
        elif m == "adapter":
            self._add_adapters()
        elif m == "prompt":
            self._add_prompt()
        elif m == "prefix":
            self._add_prefix()
        else:
            raise ValueError(f"method must be full | partial | bitfit | lorma | adapter | prompt | prefix, got {m!r}")
        if cfg.get("train_norms", False):
            for mod in bb.modules():
                if isinstance(mod, (nn.LayerNorm, nn.GroupNorm)) or type(mod).__name__.lower().endswith("rmsnorm"):
                    for p in mod.parameters(recurse=False):
                        p.requires_grad = True

    def _wrap_linears(self, kind):
        cfg = self.cfg
        targets = cfg.get("targets") or DEFAULT_TARGETS[self.name]
        r = int(cfg.get("rank", 8))
        alpha = float(cfg.get("alpha", 2 * r))
        stack_names = [BLOCKS[self.name]] + (["decoder"] if self.use_decoder else [])
        replaced = []
        for sname in stack_names:
            for bi, blk in enumerate(getattr(self.backbone, sname)):
                for path, mod in list(blk.named_modules()):
                    clean = path.replace("orig.", "").replace("inner.", "")
                    if not isinstance(mod, nn.Linear) or not any(clean == t or clean.endswith("." + t)
                                                                 for t in targets):
                        continue
                    parent_path, _, leaf = path.rpartition(".")
                    parent = blk.get_submodule(parent_path) if parent_path else blk
                    splits = _out_splits(self.name, clean, mod, self.d) if cfg.get("lorma_split", True) \
                        else [mod.out_features]
                    new = LoRMALinear(mod, r, alpha, splits, cfg.get("lorma_variant", "plus"))
                    setattr(parent, leaf, new)
                    replaced.append(f"{sname}.{bi}.{path}")
        if not replaced:
            raise ValueError(f"no linear matched targets {targets}")
        self.adapted = replaced

    def _add_adapters(self):
        m = int(self.cfg.get("adapter_size", 64))
        style = self.cfg.get("adapter_style", "houlsby")
        attn_site, mlp_site = ADAPTER_SITES[self.name]
        sites = [attn_site, mlp_site] if style == "houlsby" else [mlp_site]
        for stack in self._stacks():
            for blk in stack:
                for s in sites:
                    setattr(blk, s, WithAdapter(getattr(blk, s), self.d, m))

    def _add_prompt(self):
        n = int(self.cfg.get("n_prompt", 20))
        g = torch.Generator().manual_seed(int(self.cfg.get("init_seed", 0)))
        ids = torch.randint(1000, self.emb.num_embeddings, (n,), generator=g)
        self.prompt = nn.Parameter(self.emb.weight.detach()[ids].clone().float())
        self.n_prompt = n

    def _add_prefix(self):
        if self.name not in ("prefixlm", "encdec"):
            raise ValueError("prefix tuning is implemented for the attention models prefixlm and encdec only")
        n = int(self.cfg.get("n_prefix", 16))
        for blk in self._blocks():
            blk.attn.add_prefix(n)
        g = torch.Generator().manual_seed(int(self.cfg.get("init_seed", 0)))
        ids = torch.randint(1000, self.emb.num_embeddings, (1, n), generator=g)
        dev = self.emb.weight.device
        self.ctx.capture = True
        was = self.training
        self.eval()
        with torch.no_grad():
            self.hidden(ids.to(dev), torch.ones_like(ids, dtype=torch.bool).to(dev))
        self.ctx.capture = False
        self.train(was)

    def hidden(self, input_ids, attention_mask):
        bb = self.backbone
        B, W = input_ids.shape
        x = self.emb(input_ids)
        am = attention_mask.bool()
        if self.prompt is not None:
            x = torch.cat([self.prompt.to(x.dtype)[None].expand(B, -1, -1), x], dim=1)
            am = torch.cat([am.new_ones(B, self.n_prompt), am], dim=1)
        T = x.shape[1]
        if T > self.backbone_max_len():
            raise ValueError(f"window + prompt = {T} exceeds the Phase-1 context {self.backbone_max_len()}")
        h = getattr(self, f"_hidden_{self.name}")(x, am, T)
        return h[:, T - W:]

    def backbone_max_len(self):
        bb = self.backbone
        for n in ("max_len", "max_seq_len"):
            if isinstance(getattr(bb, n, None), int):
                return getattr(bb, n)
        c = getattr(bb, "c", None) or getattr(bb, "cfg", None) or {}
        if isinstance(c, dict) and c.get("max_seq_len"):
            return int(c["max_seq_len"])
        return 1024

    def _run(self, blk, *args):
        if self.grad_ckpt and self.training:
            return checkpoint(blk, *args, use_reentrant=False)
        return blk(*args)

    def _hidden_prefixlm(self, x, am, T):
        bb = self.backbone
        if bb.pos_emb is not None:
            x = x + bb.pos_emb(torch.arange(T, device=x.device))
        x = bb.drop(x)
        cos = bb.rope_cos[:T] if bb.rope_cos is not None else None
        sin = bb.rope_sin[:T] if bb.rope_sin is not None else None
        mask = self.p1.prefix_lm_mask(T, am.sum(1), x.device)
        for blk in bb.blocks:
            x = self._run(blk, x, cos, sin, mask)
        return bb.ln_f(x)

    def _hidden_encdec(self, x, am, T):
        bb = self.backbone
        self.ctx.key_mask = None if bool(am.all()) else am
        try:
            emb = x
            x = bb.drop(x)
            cos, sin = bb.rope_cos[:T], bb.rope_sin[:T]
            for blk in bb.encoder:
                x = self._run(blk, x, cos, sin)
            mem = bb.enc_ln(x)
            if not self.use_decoder:
                return mem
            y = bb.drop(emb)
            for blk in bb.decoder:
                y = self._run(blk.train_forward, y, cos, sin, mem)
            return bb.dec_ln(y) + mem
        finally:
            self.ctx.key_mask = None

    def _hidden_arch4(self, x, am, T):
        bb = self.backbone
        t = bb.tables(x.device)
        x = bb.drop(x)
        cos, sin, mask = t["cos"][:T], t["sin"][:T], t["mask"][:, :T, :T]
        for blk in bb.blocks:
            x = self._run(blk, x, cos, sin, mask)
        return bb.ln_f(x)

    def _hidden_biscale(self, x, am, T):
        bb = self.backbone
        drop = getattr(bb, "drop", None)
        if drop is not None:
            x = drop(x)
        for blk in bb.blocks:
            x = self._run(blk, x)
        return _first_attr(bb, ["norm", "ln_f", "final_norm"])(x)

    def forward(self, input_ids, attention_mask, doc_ids, n_docs, chunk_pos=None, chunk_rel=None):
        am = attention_mask.bool()
        h = self.hidden(input_ids, am)
        return self.head(h, am, doc_ids.long(), int(n_docs), chunk_pos, chunk_rel)

    def param_groups(self, base_lr):
        head_mult = float(self.cfg.get("head_lr_mult", 1.0))
        new_mult = float(self.cfg.get("peft_lr_mult", 1.0))
        bb_mult = float(self.cfg.get("backbone_lr_mult", 1.0))
        wd = self.cfg.get("weight_decay")
        groups = {"head": [], "head_nd": [], "new": [], "new_nd": [], "bb": [], "bb_nd": []}
        for n, p in self.named_parameters():
            if not p.requires_grad:
                continue
            nd = p.dim() < 2 or n == "prompt" or n.endswith(".pk") or n.endswith(".pv")
            if n.startswith("head."):
                key = "head"
            elif n == "prompt" or n.endswith((".pk", ".pv")) or ".A." in n or ".B." in n or ".adapter." in n:
                key = "new"
            else:
                key = "bb"
            groups[key + ("_nd" if nd else "")].append(p)
        mult = {"head": head_mult, "new": new_mult, "bb": bb_mult}
        out = []
        for k, ps in groups.items():
            if not ps:
                continue
            g = {"params": ps, "lr": base_lr * mult[k.split("_")[0]]}
            if k.endswith("_nd"):
                g["weight_decay"] = 0.0
            elif wd is not None:
                g["weight_decay"] = float(wd)
            out.append(g)
        return out


def build_cj_model(model_cfg: dict, num_labels: int) -> nn.Module:
    return CJModel(dict(model_cfg), int(num_labels))


@torch.no_grad()
def check_init_equivalence(model_cfg: dict, num_labels: int, window: int = 256, batch: int = 2, seed: int = 0):
    adapted = CJModel(dict(model_cfg), num_labels).eval()
    plain = CJModel(dict(model_cfg, method="full", keep_layers=model_cfg.get("keep_layers")), num_labels).eval()
    g = torch.Generator().manual_seed(seed)
    ids = torch.randint(1000, adapted.emb.num_embeddings, (batch, window), generator=g)
    dev = adapted.emb.weight.device
    plain.to(dev)
    am = torch.ones_like(ids, dtype=torch.bool).to(dev)
    diff = (adapted.hidden(ids.to(dev), am).float() - plain.hidden(ids.to(dev), am).float()).abs().max().item()
    n_all = sum(p.numel() for p in adapted.parameters())
    n_tr = sum(p.numel() for p in adapted.parameters() if p.requires_grad)
    print(f"{model_cfg.get('phase1')}/{model_cfg.get('method')}: max |hidden(adapted) - hidden(pretrained)| = "
          f"{diff:.2e} | trainable {n_tr/1e6:.3f}M of {n_all/1e6:.2f}M")
    return diff

Writing cj_model.py


In [18]:
print("EVAL_N" in open("lsi_train.py").read())

True


In [17]:
src = open("lsi_train.py").read()
P = [('DEV_KEYS = list(DEVS)\nY_DEV = multihot(DEV_KEYS, DEVS).astype(bool) if DEVS else None\n',
      'DEV_KEYS = list(DEVS)\nY_DEV = multihot(DEV_KEYS, DEVS).astype(bool) if DEVS else None\n'
      'EVAL_N = int(TR.get("eval_dev_cases") or 0)\n'
      'EP_KEYS = random.Random(SEED + 7).sample(DEV_KEYS, min(len(DEV_KEYS), EVAL_N)) if EVAL_N else DEV_KEYS\n'
      'Y_EP = multihot(EP_KEYS, DEVS).astype(bool) if DEVS else None\n'),
     ('        Pd = probs(predict_logits(DEVS), DEV_KEYS)\n        s5 = ld.scores(ld.decide(Pd, 0.5, MINL), Y_DEV)\n'
      '        tt, tf = ld.tune_global(Pd, Y_DEV, min_labels=MINL)',
      '        Pd = probs(predict_logits({k: DEVS[k] for k in EP_KEYS}), EP_KEYS)\n        s5 = ld.scores(ld.decide(Pd, 0.5, MINL), Y_EP)\n'
      '        tt, tf = ld.tune_global(Pd, Y_EP, min_labels=MINL)'),
     ('    every = {k: PARTS["train"][k] for k in tr_keys}; every.update(PARTS.get("dev", {}))\n',
      '    every = {k: PARTS["train"][k] for k in tr_keys}\n')]
if "EVAL_N" not in src:
    for a, b in P:
        assert src.count(a) == 1, "lsi_train.py differs — tell Claude"
        src = src.replace(a, b)
    open("lsi_train.py", "w").write(src); print("patched lsi_train.py")

patched lsi_train.py


In [19]:
# 8 — PLAN (rule 17.5: every setting below is yours). Same notebook on both accounts: set ACCOUNT, then Save & Run All.
ACCOUNT = "A"

T = dict(epochs=6, epoch_cases=8000, lr=5e-4, batch_size=4, grad_accum=4, warmup_frac=0.06, weight_decay=0.01,
         schedule="linear", loss="bce", pos_weight="balanced", pos_weight_max=10.0, threshold="per_label", min_labels=1,
         select_metric="tuned_macro_f1", max_minutes=85)
D = dict(chunk_len=512, stride=448, max_chunks=4, select="head_tail", eval_max_chunks=4, eval_select="head_tail",
         lowercase=False)

HEAD = dict(out_prior=0.039, head_hidden=512)


QUEUES = {
    "A": {0: [dict(name="r1_lorma_prefixlm", model=dict(phase1="prefixlm", method="lorma", lorma_variant="plus", rank=16, alpha=32,
                                                        pool="mean", agg="attn", head_lr_mult=2.0, **HEAD), train=T, data=D),
              dict(name="r6_kd_student", model=dict(phase1="prefixlm", method="full", keep_layers=8, keep_mode="spread",
                                                    pool="mean", agg="attn", head_lr_mult=2.0, backbone_lr_mult=0.06,
                                                    grad_ckpt=True, **HEAD), train=dict(T, max_minutes=75), data=D,
                   kd=dict(teacher="r1_lorma_prefixlm", teacher_cases=12000, temperature=2.0, alpha=0.5))],
          1: [dict(name="r2_full_encdec", model=dict(phase1="encdec", method="full", pool="mean", agg="chunk_max",
                                                     head_lr_mult=2.0, backbone_lr_mult=0.06, **HEAD), train=T, data=D),
              dict(name="r5_prompt_encdec", model=dict(phase1="encdec", method="prompt", n_prompt=32, pool="mean", agg="attn",
                                                       head_lr_mult=4.0, peft_lr_mult=10.0, **HEAD), train=T, data=D)]},
    "B": {0: [dict(name="r3_partial_arch4", model=dict(phase1="arch4", method="partial", unfreeze_layers=3, pool="mean_last",
                                                       agg="attn", head_lr_mult=2.0, backbone_lr_mult=0.2, **HEAD), train=T, data=D),
              dict(name="r7_prefix_prefixlm", model=dict(phase1="prefixlm", method="prefix", n_prefix=16, pool="mean", agg="attn",
                                                         head_lr_mult=4.0, peft_lr_mult=10.0, **HEAD), train=T, data=D)],
          1: [dict(name="r4_adapter_biscale", model=dict(phase1="biscale", method="adapter", adapter_size=64, adapter_style="houlsby",
                                                         pool="mean_last", agg="attn", head_lr_mult=2.0, peft_lr_mult=2.0,
                                                         backbone_dropout=0.0, **HEAD), train=T, data=D),
              dict(name="r8_full_prefixlm", model=dict(phase1="prefixlm", method="full", pool="mean", agg="attn", head_lr_mult=2.0,
                                                       backbone_lr_mult=0.06, grad_ckpt=True, **HEAD), train=T, data=D)]},
}
PLAN = QUEUES[ACCOUNT]
todo = [(j["name"], k) for js in PLAN.values() for j in js for part in (j["model"], j["train"], j.get("data") or {}, j.get("kd") or {})
        for k, v in part.items() if v == TODO or k == "TODO"]
assert not todo, f"fill these before Save & Run All: {todo}"

In [20]:
# 9 — model check for every planned model (shape, trainable params, gradients, memory). Failures are printed, not fatal.
import importlib, cj_model; importlib.reload(cj_model)
NL = len(LABELS)
for js in PLAN.values():
    for j in js:
        try:
            m = cj_model.build_cj_model(dict(j["model"]), NL).cuda().train()
            n_docs, per_doc, T_ = 4, 3, 512
            N = n_docs * per_doc
            x = torch.randint(5, V, (N, T_), device="cuda"); msk = torch.ones_like(x, dtype=torch.bool); msk[-1, T_ // 2:] = False
            did = torch.arange(n_docs, device="cuda").repeat_interleave(per_doc)
            pos = torch.arange(per_doc, device="cuda").repeat(n_docs); rel = pos.float() / per_doc
            torch.cuda.reset_peak_memory_stats()
            with torch.autocast("cuda", dtype=torch.float16):
                out = m(x, msk, did, n_docs, chunk_pos=pos, chunk_rel=rel)
            assert tuple(out.shape) == (n_docs, NL), f"logits must be [n_docs, {NL}], got {tuple(out.shape)}"
            out.float().mean().backward()
            nog = [n for n, p in m.named_parameters() if p.requires_grad and p.grad is None]
            tot, tr = sum(p.numel() for p in m.parameters()), sum(p.numel() for p in m.parameters() if p.requires_grad)
            print(f"OK   {j['name']:24s} trainable {tr/1e6:.3f}M of {tot/1e6:.2f}M ({100*tr/tot:.2f}%) | "
                  f"peak {torch.cuda.max_memory_allocated()/1e9:.1f} GB for 12 chunks | no-grad {nog[:3]}")
            del m, out
        except Exception as e:
            print(f"FAIL {j['name']:24s} {type(e).__name__}: {e}")
        torch.cuda.empty_cache()

OK   r1_lorma_prefixlm        trainable 3.101M of 112.73M (2.75%) | peak 3.5 GB for 12 chunks | no-grad []
OK   r6_kd_student            trainable 81.728M of 81.73M (100.00%) | peak 0.8 GB for 12 chunks | no-grad []
OK   r2_full_encdec           trainable 67.551M of 67.55M (100.00%) | peak 1.6 GB for 12 chunks | no-grad []
OK   r5_prompt_encdec         trainable 0.472M of 67.58M (0.70%) | peak 1.3 GB for 12 chunks | no-grad []


In [21]:
# 10 — RUN THE PLAN (blocks until every queue on this account is done; progress prints as runs finish)
run_queues(PLAN)
summary([j["name"] for js in PLAN.values() for j in js])

[   0.0 min] started r1_lorma_prefixlm on GPU 0
[   0.0 min] started r2_full_encdec on GPU 1

==== [  47.0 min] r2_full_encdec exit 0 (47.0 min)
  best so far (epoch 3)
epoch 4 | train_loss 0.5452 | minutes 25.5295 | dev_macro_f1 0.1095 | dev_micro_f1 0.2772 | dev_tuned_macro_f1 0.1146 | dev_tuned_t 0.3900 | dev_labels_per_doc 8.2490
  best so far (epoch 4)
epoch 5 | train_loss 0.5316 | minutes 32.5517 | dev_macro_f1 0.1209 | dev_micro_f1 0.2887 | dev_tuned_macro_f1 0.1271 | dev_tuned_t 0.4100 | dev_labels_per_doc 10.7691
  best so far (epoch 5)
epoch 6 | train_loss 0.5187 | minutes 39.5678 | dev_macro_f1 0.1252 | dev_micro_f1 0.3023 | dev_tuned_macro_f1 0.1331 | dev_tuned_t 0.4100 | dev_labels_per_doc 8.4684
  best so far (epoch 6)
BEST on dev (threshold 0.5) macro-F1 (labels in gold) 0.1252 [100 labels] | macro-F1 (all labels) 0.1252 | micro-F1 0.3023 (P 0.221 R 0.478) | subset acc 0.0011 | labels/doc pred 8.47 gold 3.92 | docs with no prediction 0 | n 10181
BEST on dev (global thres

In [22]:
# 11 — results table (report) + download links + small backup zip (no weights)
import pandas as pd
from IPython.display import FileLink, display
rows = []
for rp in sorted(glob.glob(f"{R}/*/*_result.json")):
    r = json.load(open(rp))
    if r["run"].startswith("smoke"): continue
    dv = r.get("dev", {})
    rows.append(dict(run=r["run"], dev_macro_f1=dv.get("macro_f1_present"), dev_micro_f1=dv.get("micro_f1"),
                     dev_subset_acc=dv.get("subset_acc"), threshold=r.get("threshold_mode"), loss=r.get("loss"),
                     pos_weight=r.get("pos_weight"), trainable_M=r["params_trainable"]/1e6, total_M=r["params_total"]/1e6,
                     train_min=r["train_minutes"], peak_GB=r["peak_mem_gb"], epochs_done=len(r.get("history", [])),
                     chunk_len=r["chunk_len"], max_chunks=r["max_chunks"], select=r["select"]))
df = pd.DataFrame(rows).sort_values("run"); display(df)
df.to_csv(f"{R}/results.csv", index=False)
os.chdir("/kaggle/working")
for n in df.run: display(FileLink(f"p5runs/{n}/{n}_submission.zip"))
display(FileLink("p5runs/results.csv"))
!zip -qr p5_backup.zip cj_model.py p5runs -x "*.pt" "*.npz" "*smoke*"
display(FileLink("p5_backup.zip"))

,run,dev_macro_f1,dev_micro_f1,dev_subset_acc,threshold,loss,pos_weight,trainable_M,total_M,train_min,peak_GB,epochs_done,chunk_len,max_chunks,select
0,r1_lorma_prefixlm,0.356879,0.448072,0.023475,per_label,bce,balanced,3.101285,112.733285,87.615072,4.400396,6,512,4,head_tail
1,r1_lorma_prefixlm_teacher,0.356879,0.448072,0.023475,per_label,bce,balanced,3.101285,112.733285,0.000024,0.452245,0,512,4,head_tail
2,r2_full_encdec,0.182827,0.232644,0.000982,per_label,bce,balanced,67.551332,67.551332,41.769998,2.837000,6,512,4,head_tail
3,r5_prompt_encdec,0.110480,0.140089,0.000000,per_label,bce,balanced,0.471653,67.576421,37.858908,1.648284,6,512,4,head_tail
4,r6_kd_student,0.364264,0.457696,0.030154,per_label,bce,balanced,81.727589,81.727589,60.977930,1.894791,6,512,4,head_tail


/kaggle/working/p5runs/r1_lorma_prefixlm/r1_lorma_prefixlm_submission.zip

/kaggle/working/p5runs/r1_lorma_prefixlm_teacher/r1_lorma_prefixlm_teacher_submission.zip

/kaggle/working/p5runs/r2_full_encdec/r2_full_encdec_submission.zip

/kaggle/working/p5runs/r5_prompt_encdec/r5_prompt_encdec_submission.zip

/kaggle/working/p5runs/r6_kd_student/r6_kd_student_submission.zip

/kaggle/working/p5runs/results.csv

/kaggle/working/p5_backup.zip